In [3]:
import pandas as pd

df = pd.read_csv("dSets/synthetic_orders_model.csv")

df["has_return_history"] = df["customer_past_return_rate"].notna().astype(int)

mean_return_rate = df["customer_past_return_rate"].mean()
df["customer_past_return_rate"] = df["customer_past_return_rate"].fillna(mean_return_rate)

print(f"Population mean return rate used for imputation: {mean_return_rate:.4f}")
print(f"Orders with known history: {df['has_return_history'].sum()} / {len(df)}")

categorical_cols = ["category", "payment_method", "delivery_pincode_tier", "time_of_day_ordered"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

id_cols = ["order_id", "customer_id", "order_date"]
df_encoded = df_encoded.drop(columns=id_cols)

print(f"\nShape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nColumns after encoding:\n{list(df_encoded.columns)}")

df_encoded.to_csv("dSets/synthetic_orders_prepped.csv", index=False)
print("\nSaved: dSets/synthetic_orders_prepped.csv")

Population mean return rate used for imputation: 0.2500
Orders with known history: 4338 / 5000

Shape before encoding: (5000, 16)
Shape after encoding: (5000, 25)

Columns after encoding:
['order_value', 'discount_pct', 'is_first_time_buyer', 'customer_past_orders', 'customer_past_return_rate', 'size_variant_flag', 'days_to_deliver', 'returned', 'has_return_history', 'category_beauty', 'category_electronics', 'category_fashion', 'category_grocery', 'category_home', 'payment_method_COD', 'payment_method_UPI', 'payment_method_card', 'payment_method_netbanking', 'delivery_pincode_tier_metro', 'delivery_pincode_tier_tier2', 'delivery_pincode_tier_tier3', 'time_of_day_ordered_afternoon', 'time_of_day_ordered_evening', 'time_of_day_ordered_morning', 'time_of_day_ordered_night']

Saved: dSets/synthetic_orders_prepped.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv("dSets/synthetic_orders_prepped.csv")

X = df.drop(columns=["returned"])
y = df["returned"]

X_trainpool, X_test, y_trainpool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training pool: {X_trainpool.shape[0]} orders, return rate: {y_trainpool.mean():.3f}")
print(f"Held-out test: {X_test.shape[0]} orders, return rate: {y_test.mean():.3f}")

X_test.to_csv("dSets/X_test_FINAL.csv", index=False)
y_test.to_csv("dSets/y_test_FINAL.csv", index=False)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

gb_model = GradientBoostingClassifier(random_state=42)

scoring = ["roc_auc", "average_precision"]  # average_precision = PR-AUC

print("\n--- Logistic Regression (5-fold CV) ---")
log_reg_scores = cross_validate(log_reg_pipeline, X_trainpool, y_trainpool, cv=cv, scoring=scoring)
print(f"ROC-AUC: {log_reg_scores['test_roc_auc'].mean():.4f} ± {log_reg_scores['test_roc_auc'].std():.4f}")
print(f"PR-AUC:  {log_reg_scores['test_average_precision'].mean():.4f} ± {log_reg_scores['test_average_precision'].std():.4f}")

print("\n--- Gradient Boosting (5-fold CV) ---")
gb_scores = cross_validate(gb_model, X_trainpool, y_trainpool, cv=cv, scoring=scoring)
print(f"ROC-AUC: {gb_scores['test_roc_auc'].mean():.4f} ± {gb_scores['test_roc_auc'].std():.4f}")
print(f"PR-AUC:  {gb_scores['test_average_precision'].mean():.4f} ± {gb_scores['test_average_precision'].std():.4f}")

X_trainpool.to_csv("dSets/X_trainpool.csv", index=False)
y_trainpool.to_csv("dSets/y_trainpool.csv", index=False)

Training pool: 4000 orders, return rate: 0.200
Held-out test: 1000 orders, return rate: 0.200

--- Logistic Regression (5-fold CV) ---
ROC-AUC: 0.7481 ± 0.0244
PR-AUC:  0.4463 ± 0.0378

--- Gradient Boosting (5-fold CV) ---
ROC-AUC: 0.7384 ± 0.0165
PR-AUC:  0.4350 ± 0.0316


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

X_trainpool = pd.read_csv("dSets/X_trainpool.csv")
y_trainpool = pd.read_csv("dSets/y_trainpool.csv").squeeze()
X_test = pd.read_csv("dSets/X_test_FINAL.csv")
y_test = pd.read_csv("dSets/y_test_FINAL.csv").squeeze()

log_reg_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
log_reg_final.fit(X_trainpool, y_trainpool)

gb_final = GradientBoostingClassifier(random_state=42)
gb_final.fit(X_trainpool, y_trainpool)

log_reg_test_probs = log_reg_final.predict_proba(X_test)[:, 1]
gb_test_probs = gb_final.predict_proba(X_test)[:, 1]

print("=== FINAL HELD-OUT TEST SET RESULTS (evaluated once) ===\n")

print("Logistic Regression (PRIMARY MODEL):")
print(f"  ROC-AUC: {roc_auc_score(y_test, log_reg_test_probs):.4f}")
print(f"  PR-AUC:  {average_precision_score(y_test, log_reg_test_probs):.4f}")

print("\nGradient Boosting (comparison model):")
print(f"  ROC-AUC: {roc_auc_score(y_test, gb_test_probs):.4f}")
print(f"  PR-AUC:  {average_precision_score(y_test, gb_test_probs):.4f}")

pd.DataFrame({
    "y_true": y_test,
    "log_reg_prob": log_reg_test_probs,
    "gb_prob": gb_test_probs,
}).to_csv("dSets/test_predictions.csv", index=False)

print("\nSaved: dSets/test_predictions.csv")

=== FINAL HELD-OUT TEST SET RESULTS (evaluated once) ===

Logistic Regression (PRIMARY MODEL):
  ROC-AUC: 0.7710
  PR-AUC:  0.4753

Gradient Boosting (comparison model):
  ROC-AUC: 0.7535
  PR-AUC:  0.4728

Saved: test_predictions.csv
